# Notebook 09b — Robustness II: re-routing sweeps ($\lambda_{stress}$, $k$, $\beta_{grade}$, flat-speed ablation)

*Thesis §Model Robustness, items 3–6.*

Unlike 09a, every setting here changes the **routing**, so each needs fresh $k$-shortest paths — this
is the slow robustness notebook. Three features keep it tractable and faithful to the main run:

1. **Corridor-member station matching.** Paths map to stations through Notebook 04's corridor members
   (`count_link_members.csv`), exactly as Notebook 05 does, so the baseline row reproduces the frozen
   Stage-3 $\pi^*$ from Notebook 06.
2. **Cached, shared path stores.** Each store keeps $K_{STORE}=5$ paths, so the whole $k$ sweep and
   the baseline rows of the $\lambda$ and $\beta$ sweeps reuse Notebook 05's routing. $\lambda=1$ reuses
   the stress-tolerant store (identical cost). Fresh routing is needed only for $\lambda\in\{1.4,4,8.2\}$,
   $\beta\in\{12,59\}$ (×2 classes), and the flat-speed pair.
3. **Parallel routing** across `N_JOBS` cores (results identical to a serial run).

**Decisions use training data only** (training mean GEH and 5-fold training-only CV mean GEH at each
setting's own $\pi^*$). Held-out stations are scored once per setting and reported only. Every sweep
asserts that its baseline row reproduces Notebook 06's $\pi^*$.

Each setting starts its outer loop from the Stage-1 demand $t$ of Notebook 04 (Stage 1 uses the neutral
time cost, independent of $\lambda$ and $k$); the outer loop then re-estimates $t$ at that setting.

**Outputs:** `robustness_k.csv`, `robustness_stress_penalty.csv`, `robustness_beta_grade.csv`,
`robustness_flat.csv`, `09b_sweeps.png`, `reports/09b_robustness_report.txt`.


In [7]:
import os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
import thesis_common as tc

tc.set_style()
NB = "09b"
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
# Toggle individual sweeps + the CV pass. Set a sweep False to skip it (e.g. to re-draw figures fast).
RUN_STRESS, RUN_K, RUN_BETA, RUN_FLAT, RUN_CV = True, True, True, True, True

# lambda_stress grid: 1 + Broach (2012) commute marginal rates of substitution by AADT band
# (10-20k: 1.37, 20-30k: 2.40, 30k+: 8.16). 4.0 is an interpolated midpoint between Broach's bands.
STRESS_PENALTIES = [1.0, 1.4, 2.4, 4.0, 8.2]
K_VALUES = [1, 2, 3, 4, 5]
# beta_grade: metres of flat-equivalent distance per metre climbed. Broach commute 2-4% -> ~12;
# Hood et al. (2011) -> ~59.
BETA_GRADE = [0.0, 12.0, 59.0]
print("N_JOBS", N_JOBS)


N_JOBS 9


In [8]:
# Load the frozen main-run context and define the per-setting helpers.
final = tc.read_final(); PI_REF = final["pi_star"]
cal   = pd.read_csv(tc.DATA_PROCESSED / "od_calibrated_t.csv").sort_values("pair_idx").reset_index(drop=True)
links = pd.read_csv(tc.DATA_PROCESSED / "A_neutral_links.csv").sort_values("link_idx").reset_index(drop=True)
snap  = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
L = len(links); v_obs = links.v_obs.to_numpy(float)
n_seed, t_cal = cal.n_seed.to_numpy(float), cal.t.to_numpy(float)
pairs = list(zip(cal.o_node, cal.d_node))
mon = tc.load_members(L)
P = len(cal)
exp_pairs, assembly, rep_idx = tc.expand_pairs(cal)   # identical expansion to NB05 (shared cache)
train = tc.load_train_mask(L); folds = tc.make_cv_folds(train, v_obs)
W, _ = tc.build_W(tc.load_survey(), tc.site_to_link(snap, links), L)
Gr = tc.load_network()
A_ref_low = sparse.load_npz(tc.DATA_PROCESSED / "A_low.npz"); A_ref_high = sparse.load_npz(tc.DATA_PROCESSED / "A_high.npz")

def stores(lam=tc.STRESS_PENALTY, beta=0.0, flat=False):
    """Route (or load cached) the high- and low-stress path stores for one cost setting.
    lam == 1 makes the low-stress cost identical to the high-stress cost, so the same store is reused."""
    tc.set_costs(Gr, lam=lam, beta=beta, flat=flat)
    sh = tc.route_store(tc.to_simple(Gr, "cost_high"), exp_pairs, tc.K_STORE, "cost_high",
                        tc.store_tag("high", beta=beta, flat=flat), n_jobs=N_JOBS)
    if abs(lam - 1.0) < 1e-12:
        return sh, sh
    sl = tc.route_store(tc.to_simple(Gr, "cost_low"), exp_pairs, tc.K_STORE, "cost_low",
                        tc.store_tag("low", lam=lam, beta=beta, flat=flat), n_jobs=N_JOBS)
    return sl, sh

def evaluate(sl, sh, k, tag):
    """Assemble A_low/A_high at k paths, run the Stage-3 outer loop, and return the fit metrics
    (training, training-only CV, and reported-only held-out) plus the two matrices."""
    A_l, A_h = (tc.assemble_A_spread(sl, k, mon, L, assembly, P),
                tc.assemble_A_spread(sh, k, mon, L, assembly, P))
    r = tc.outer_loop(A_l, A_h, t_cal, n_seed, v_obs, W, train)
    f = tc.flow(A_l, A_h, r["pi_star"], r["t_star"])
    mtr = tc.metrics(f[train], v_obs[train])
    out = dict(pi_star=r["pi_star"], n_iter=r["n_iter"], converged=r["reason"], a_diff=tc.a_diff(A_l, A_h),
               train_mean_geh=mtr["mean_geh"], train_geh_pass=mtr["geh_pass"], train_rmse=mtr["rmse"], train_r2=mtr["r2"],
               pi_rmse=tc.objective_curves(A_l, A_h, r["t_star"], v_obs, train)["pi_rmse"])
    if RUN_CV:
        out["cv_mean_geh"], _, pis = tc.cv_score(A_l, A_h, t_cal, n_seed, v_obs, W, train, folds)
        out["cv_pi_range"] = f"{min(pis):.2f}-{max(pis):.2f}"
    test = tc.load_test_mask(L, f"robustness {tag}", "NB09b")       # reported only
    mte = tc.metrics(f[test], v_obs[test])
    out.update(test_mean_geh=mte["mean_geh"], test_geh_pass=mte["geh_pass"], test_rmse=mte["rmse"], test_r2=mte["r2"])
    print(f"  {tag:<16} pi*={out['pi_star']:.2f}  |A_l-A_h| {out['a_diff']:.0%}  train meanGEH {out['train_mean_geh']:.2f}  "
          f"CV {out.get('cv_mean_geh', np.nan):.2f}  | held-out R2 {out['test_r2']:.2f} (reported only)")
    return out, A_l, A_h

def check_baseline(df, name):
    """Assert the sweep's baseline row reproduces the frozen Stage-3 pi* (anchors the sweep)."""
    b = df[df.is_baseline.astype(bool)]
    assert len(b) == 1 and abs(b.pi_star.iloc[0] - PI_REF) < 1e-9, (
        f"{name}: baseline gives pi*={b.pi_star.iloc[0]:.2f}, NB07 froze {PI_REF:.2f} - inputs out of sync")
    print(f"  {name}: baseline reproduces NB07 pi* = {PI_REF:.2f}")


## Baseline routing (from the Notebook 05 cache) and the $k$ sweep

Loads the baseline path stores (cache hit, no re-routing), asserts the assembled baseline matrices are
identical to Notebook 06's `A_low`/`A_high`, then sweeps the number of stored paths used per O–D pair,
$k=1\dots5$. Because 5 paths are cached, the whole $k$ sweep reuses the same routing.


In [9]:
t0 = time.time()
SL0, SH0 = stores()
A_l3, A_h3 = (tc.assemble_A_spread(SL0, tc.K_PATHS, mon, L, assembly, P),
              tc.assemble_A_spread(SH0, tc.K_PATHS, mon, L, assembly, P))
# Sanity: the baseline-k matrices must match the frozen NB06 ones exactly.
assert abs(A_l3 - A_ref_low).sum() < 1e-9 and abs(A_h3 - A_ref_high).sum() < 1e-9, \
    "baseline matrices differ from NB06's A_low/A_high - re-run NB06"
print("baseline matrices identical to NB06")
if RUN_K:
    rows = []
    for k in K_VALUES:
        r, *_ = evaluate(SL0, SH0, k, f"k={k}"); r.update(param="k", value=k, label=str(k), is_baseline=(k == tc.K_PATHS)); rows.append(r)
    kdf = pd.DataFrame(rows); check_baseline(kdf, "k sweep"); kdf.to_csv(tc.DATA_PROCESSED / "robustness_k.csv", index=False)


  [high_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  [low_lam2.4_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
baseline matrices identical to NB06
  k=1              pi*=0.68  |A_l-A_h| 123%  train meanGEH 4.90  CV 7.02  | held-out R2 0.56 (reported only)
  k=2              pi*=0.67  |A_l-A_h| 122%  train meanGEH 4.92  CV 7.11  | held-out R2 0.55 (reported only)
  k=3              pi*=0.64  |A_l-A_h| 120%  train meanGEH 4.98  CV 7.25  | held-out R2 0.55 (reported only)
  k=4              pi*=0.63  |A_l-A_h| 120%  train meanGEH 4.99  CV 7.25  | held-out R2 0.55 (reported only)
  k=5              pi*=0.66  |A_l-A_h| 120%  train meanGEH 4.98  CV 7.21  | held-out R2 0.56 (reported only)
  k sweep: baseline reproduces NB07 pi* = 0.64


## $\lambda_{stress}$ sweep

Varies the stress penalty applied to links outside the low-stress set. At $\lambda=1$ the two classes
become identical ($|A_{low}-A_{high}|=0$), so $\pi$ is unidentified by construction — included as the
degenerate reference point. Fresh routing is needed only for the non-baseline $\lambda$ values.


In [10]:
if RUN_STRESS:
    rows = []
    for lam in STRESS_PENALTIES:
        sl, sh = (SL0, SH0) if lam == tc.STRESS_PENALTY else stores(lam=lam)
        r, *_ = evaluate(sl, sh, tc.K_PATHS, f"lambda={lam:g}")
        r.update(param="lambda_stress", value=lam, label=f"{lam:g}", is_baseline=(lam == tc.STRESS_PENALTY)); rows.append(r)
        pd.DataFrame(rows).to_csv(tc.DATA_PROCESSED / "robustness_stress_penalty.csv", index=False)   # checkpoint as we go
    ldf = pd.DataFrame(rows); check_baseline(ldf, "lambda sweep")
    print("  note: at lambda=1 the classes are identical (|A_l-A_h|=0), so pi is unidentified by construction")


  [high_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  lambda=1         pi*=0.00  |A_l-A_h| 0%  train meanGEH 6.38  CV 8.97  | held-out R2 0.33 (reported only)
  [high_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  [low_lam1.4_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  lambda=1.4       pi*=0.63  |A_l-A_h| 78%  train meanGEH 5.37  CV 7.98  | held-out R2 0.51 (reported only)
  lambda=2.4       pi*=0.64  |A_l-A_h| 120%  train meanGEH 4.98  CV 7.25  | held-out R2 0.55 (reported only)
  [high_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  [low_lam4_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  lambda=4         pi*=0.71  |A_l-A_h| 139%  train meanGEH 5.11  CV 7.41  | held-out R2 0.57 (reported only)
  [high_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  [low_lam8.2_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  lambda=8.2       pi*=0.67  |A_l

## $\beta_{grade}$ sweep (additive climb penalty in both class costs)

Adds an uphill climb penalty to both class costs ($\beta$ = metres of flat-equivalent distance per metre
climbed) and re-routes. Shows how strongly an explicit gradient penalty, over and above the
grade-adjusted speed, would move the share.


In [11]:
if RUN_BETA:
    rows = []
    for beta in BETA_GRADE:
        sl, sh = (SL0, SH0) if beta == 0 else stores(beta=beta)
        r, *_ = evaluate(sl, sh, tc.K_PATHS, f"beta={beta:g}")
        r.update(param="beta_grade", value=beta, label=f"{beta:g}", is_baseline=(beta == 0)); rows.append(r)
        pd.DataFrame(rows).to_csv(tc.DATA_PROCESSED / "robustness_beta_grade.csv", index=False)
    bdf = pd.DataFrame(rows); check_baseline(bdf, "beta sweep")


  beta=0           pi*=0.64  |A_l-A_h| 120%  train meanGEH 4.98  CV 7.25  | held-out R2 0.55 (reported only)
  [high_beta12_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  [low_lam2.4_beta12_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  beta=12          pi*=0.89  |A_l-A_h| 119%  train meanGEH 5.13  CV 7.52  | held-out R2 0.54 (reported only)
  [high_beta59_grade_k5_sprd5] cache signature differs -> re-routing
    [high_beta59_grade_k5_sprd5] 200/8,721    3.3 min  (~142 min left)
    [high_beta59_grade_k5_sprd5] 400/8,721    3.7 min  (~76 min left)
    [high_beta59_grade_k5_sprd5] 600/8,721    4.7 min  (~64 min left)
    [high_beta59_grade_k5_sprd5] 800/8,721    7.1 min  (~71 min left)
    [high_beta59_grade_k5_sprd5] 1,000/8,721   10.8 min  (~83 min left)
    [high_beta59_grade_k5_sprd5] 1,200/8,721   13.1 min  (~82 min left)
    [high_beta59_grade_k5_sprd5] 1,400/8,721   15.7 min  (~82 min left)
    [high_beta59_grade_k5_sprd5] 1,600/8,721   17.1 min  (~7

## Grade ablation through $\pi^*$ (flat speed in both classes)

Re-routes both classes on a flat-speed cost ($v_e=v_{flat}$, no elevation term) and re-estimates, so the
effect of grade-adjusted routing can be read end to end at $\pi^*$ rather than only at Stage 1 (which is
what Notebook 04's ablation shows).


In [12]:
if RUN_FLAT:
    rows = []
    for flat in (False, True):
        sl, sh = (SL0, SH0) if not flat else stores(flat=True)
        r, *_ = evaluate(sl, sh, tc.K_PATHS, "flat speed" if flat else "grade-adjusted")
        r.update(param="speed", value=int(flat), label="flat" if flat else "grade-adjusted", is_baseline=not flat); rows.append(r)
    fdf = pd.DataFrame(rows); check_baseline(fdf, "grade ablation"); fdf.to_csv(tc.DATA_PROCESSED / "robustness_flat.csv", index=False)
print(f"total wall time {(time.time()-t0)/60:.1f} min")


  grade-adjusted   pi*=0.64  |A_l-A_h| 120%  train meanGEH 4.98  CV 7.25  | held-out R2 0.55 (reported only)
  [high_beta0_flat_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  [low_lam2.4_beta0_flat_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  flat speed       pi*=0.67  |A_l-A_h| 112%  train meanGEH 5.30  CV 7.69  | held-out R2 0.53 (reported only)
  grade ablation: baseline reproduces NB07 pi* = 0.64
total wall time 298.6 min


## Figures and report

Draws one twin-axis panel per sweep ($\pi^*$ against training and CV fit) and writes the combined report.
The $\lambda$, $k$, $\beta$ and flat-speed rows feed thesis Table 4.3; the printed report block is this
notebook's end-of-run key-results summary.


In [13]:
# Collect whichever sweep CSVs exist, draw one panel each, and write the combined text report.
sweeps = [(n, tc.DATA_PROCESSED / f) for n, f in [("k", "robustness_k.csv"), ("lambda_stress", "robustness_stress_penalty.csv"),
                                                   ("beta_grade", "robustness_beta_grade.csv"), ("speed", "robustness_flat.csv")]]
sweeps = [(n, pd.read_csv(p)) for n, p in sweeps if p.exists()]
fig, axes = plt.subplots(1, len(sweeps), figsize=(4.2 * len(sweeps), 3.8), squeeze=False)
for ax, (n, d) in zip(axes[0], sweeps):
    x = np.arange(len(d))
    ax.plot(x, d.pi_star, "o-", color="black"); ax.axhline(PI_REF, color=tc.COL["low"], ls="--", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(d.label); ax.set_ylim(0, 1); ax.set_xlabel(n); ax.set_ylabel(r"$\pi^*$")
    axr = ax.twinx(); axr.spines["right"].set_visible(True)
    axr.plot(x, d.train_mean_geh, "s--", color=tc.COL["train"])
    if "cv_mean_geh" in d: axr.plot(x, d.cv_mean_geh, "^:", color=tc.COL["train"], alpha=0.6)
    axr.set_ylabel("train / CV mean GEH", color=tc.COL["train"])
fig.suptitle("Re-routing sweeps: $\\pi^*$ (black) and training-data fit (blue: train, dotted: CV)", y=1.04)
tc.save_diag(fig, NB, "sweeps")

cols = ["label", "pi_star", "a_diff", "n_iter", "train_mean_geh", "cv_mean_geh", "test_mean_geh", "test_geh_pass", "test_r2"]
rep = [f"09b - robustness (re-routing) ({tc.RUN_STAMP})", "=" * 60,
       f"baseline: lambda={tc.STRESS_PENALTY}, k={tc.K_PATHS}, beta=0, grade-adjusted; NB07 pi*={PI_REF:.2f}",
       "Decision columns: train_mean_geh, cv_mean_geh (training only). Held-out columns: reported only.", ""]
for n, d in sweeps:
    rep += [f"{n.upper()}  (pi* span {np.ptp(d.pi_star):.2f})", d[[c for c in cols if c in d]].round(3).to_string(index=False), ""]
rep = "\n".join(rep); (tc.REPORTS / "09b_robustness_report.txt").write_text(rep, encoding="utf-8"); print(rep)


  saved diagnostic     09b_sweeps.png
09b - robustness (re-routing) (20261028_1755)
baseline: lambda=2.4, k=3, beta=0, grade-adjusted; NB07 pi*=0.64
Decision columns: train_mean_geh, cv_mean_geh (training only). Held-out columns: reported only.

K  (pi* span 0.05)
 label  pi_star  a_diff  n_iter  train_mean_geh  cv_mean_geh  test_mean_geh  test_geh_pass  test_r2
     1     0.68   1.226      10           4.899        7.019          5.614         45.833    0.560
     2     0.67   1.225       9           4.918        7.106          5.668         41.667    0.551
     3     0.64   1.204       8           4.977        7.249          5.617         41.667    0.545
     4     0.63   1.198       9           4.992        7.254          5.476         41.667    0.554
     5     0.66   1.203      10           4.984        7.206          5.493         45.833    0.558

LAMBDA_STRESS  (pi* span 0.71)
 label  pi_star  a_diff  n_iter  train_mean_geh  cv_mean_geh  test_mean_geh  test_geh_pass  test_r2
   